# ClimateTwin AI — Exploratory Data Analysis & Preprocessing Walkthrough
**Owner:** Lokesh Joshi | AI/ML Lead (Team PrithviNex)  
**Pilot Location:** Pune, Maharashtra (`18.5204° N, 73.8567° E`, Elevation `560 m`)

This notebook explores the multi-year environmental dataset (`2022-01-01` to `2026-10-31`, 1,765 daily records), validates physical bounds, analyzes missingness and seasonal patterns, and verifies leak-free chronological preprocessing.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from ml_engine.reference_data import load_or_create_pilot_dataset
from ml_engine.data_preprocessing import EnvironmentalDataPreprocessor
from ml_engine.feature_engineering import EnvironmentalFeatureEngineer

raw_df = load_or_create_pilot_dataset()
print(f'Loaded {len(raw_df)} daily records for {raw_df["location"].unique()}')
raw_df.head()

### Analysis of Schema & Seasonal Climatology
The dataset spans `1,765` daily observations across four complete seasonal cycles in Pune (Pre-Monsoon hot dry summer Mar–May, SW Monsoon heavy rainfall Jun–Sep, Post-Monsoon transition Oct–Nov, and Winter radiative inversion Dec–Feb). Next, we fit the `EnvironmentalDataPreprocessor` strictly on the chronological training split (`first 70%`) to prevent data leakage.

In [2]:
train_end = int(len(raw_df) * 0.70)
preprocessor = EnvironmentalDataPreprocessor(preserve_missing_indicators=True)
preprocessor.fit(raw_df.iloc[:train_end])
clean_df, prep_report = preprocessor.transform(raw_df, return_report=True)

engineer = EnvironmentalFeatureEngineer()
feat_df, feat_report = engineer.transform(clean_df, return_report=True)
print('Data coverage (%):', prep_report['data_coverage_pct'])
print('Derived features created:', feat_report['derived_feature_count'])

### Summary & Conclusion
1. **Leak-Free Preprocessing:** All imputation medians (`monthly_feature_medians_`) and IQR/Z-score outlier thresholds were fitted strictly on the first 70% of chronological records.
2. **Extreme Weather Preservation:** Genuine monsoon downpours (`>115 mm/day`) and summer heatwave spikes (`>42°C`) are flagged via `is_climate_extreme` rather than deleted.
3. **Causal Feature Engineering:** All 39 engineered features (`heat_index_c`, `wet_bulb_temp_c`, `runoff_potential_index`, `cpcb_calculated_aqi`, `hydro_water_deficit_30d_mm`) use strictly trailing causal windows.